# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |TS008_Gender |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |TS008_Gender|
| Output File | gender_df|
| Notebook Author | Peter |
| Last Updated |2026-07-11 |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [3]:
# File paths

input_file = "RM121_Gender.xlsx"
output_file = "gender.json"

raw_data_path = Path("../../data/demographic/raw/gender")
output_data_path = Path("../../data/demographic/processed/gender")

## 3. Load Dataset

In [4]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Sex (2 categories) Code,Sex (2 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,1,Female,0,Aged under 1 year,449
1,E06000001,Hartlepool,1,Female,1,Aged 1 year,458
2,E06000001,Hartlepool,1,Female,2,Aged 2 years,510
3,E06000001,Hartlepool,1,Female,3,Aged 3 years,476
4,E06000001,Hartlepool,1,Female,4,Aged 4 years,518


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [5]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56932 entries, 0 to 56931
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Lower tier local authorities Code  56932 non-null  object
 1   Lower tier local authorities       56932 non-null  object
 2   Sex (2 categories) Code            56932 non-null  int64 
 3   Sex (2 categories)                 56932 non-null  object
 4   Age (86 categories) Code           56932 non-null  int64 
 5   Age (86 categories)                56932 non-null  object
 6   Observation                        56932 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 3.0+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Sex (2 categories) Code', 'Sex (2 categories)',
       'Age (86 categories) Code', 'Age (86 categories)', 'Observation'],
      dtype='object')

In [6]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Sex (2 categories) Code,Sex (2 categories),Age (86 categories) Code,Age (86 categories),Observation
count,56932,56932,56932.000000,56932,56932.000000,56932,56932.000000
unique,331,331,NaN,2,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Female,NaN,Aged under 1 year,NaN
freq,172,172,NaN,28466,NaN,662,NaN
mean,NaN,NaN,1.500000,NaN,42.500000,NaN,1046.819557
std,NaN,NaN,0.500004,NaN,24.824601,NaN,819.675209
min,NaN,NaN,1.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,1.000000,NaN,21.000000,NaN,559.000000
50%,NaN,NaN,1.500000,NaN,42.500000,NaN,792.000000
75%,NaN,NaN,2.000000,NaN,64.000000,NaN,1268.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [7]:
# Cleaning code
selected_columns = ['Lower tier local authorities',
                    'Sex (2 categories)',
                    'Age (86 categories) Code',
                    'Observation']

df_processed = df_processed[selected_columns]
df_processed.head()

,Lower tier local authorities,Sex (2 categories),Age (86 categories) Code,Observation
0,Hartlepool,Female,0,449
1,Hartlepool,Female,1,458
2,Hartlepool,Female,2,510
3,Hartlepool,Female,3,476
4,Hartlepool,Female,4,518


## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [8]:
# Dataset-specific processing
def age_group(age):
    if age <= 4:
        return "0-4"
    elif age <= 11:
        return "5-11"
    elif age <= 17:
        return "12-17"
    elif age <= 22:
        return "18-22"
    elif age <= 34:
        return "23-34"
    elif age <= 49:
        return "35-49"
    elif age <= 64:
        return "50-64"
    elif age <= 74:
        return "65-74"
    else:
        return "75+"
    
age_order = [ "0-4",
             "5-11",
             "12-17",
             "18-22",
             "23-34",
             "35-49",
             "50-64",
             "65-74",
             "75+"]

In [9]:
def get_gender_distribution(local_authority):
    processed_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    processed_data['Age Group'] = processed_data['Age (86 categories) Code'].apply(age_group)

    result = (processed_data.groupby(['Lower tier local authorities', 'Age Group'], as_index=False)['Observation'].sum())

    result['Age Group'] = pd.Categorical(result['Age Group'],categories=age_order,ordered=True)

    result = result.sort_values('Age Group')

    result['Percentage'] = (result['Observation'] / result['Observation'].sum())

    result = result.reset_index(drop=True)

    return result

In [10]:
get_gender_distribution("Manchester")

,Lower tier local authorities,Age Group,Observation,Percentage
0,Manchester,0-4,34378,0.062286
1,Manchester,5-11,51349,0.093034
2,Manchester,12-17,41225,0.074691
3,Manchester,18-22,61783,0.111938
4,Manchester,23-34,123441,0.223649
5,Manchester,35-49,108892,0.197290
6,Manchester,50-64,78703,0.142593
7,Manchester,65-74,30300,0.054897
8,Manchester,75+,21869,0.039622


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [11]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56932 entries, 0 to 56931
Data columns (total 4 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Lower tier local authorities  56932 non-null  object
 1   Sex (2 categories)            56932 non-null  object
 2   Age (86 categories) Code      56932 non-null  int64 
 3   Observation                   56932 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1.7+ MB


Lower tier local authorities    0
Sex (2 categories)              0
Age (86 categories) Code        0
Observation                     0
dtype: int64

## 8. Export Processed Dataset

In [12]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

gender_df = get_gender_distribution("Manchester")

gender_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported